In [ ]:
#| default_exp ops

# ops

> Standard async operations on a running kernel, for `ConKernelClient`

`conkernelclient.core` makes concurrent `execute()` calls safe; this module adds the operations every client ends up needing on top of that: request-scoped iopub collection, execute-and-drain composites, nbformat-style output conversion, interrupt, generic shell/control requests, and a kernel lifecycle context manager. It distills helpers that grew up independently in ipymini's test suite and solveit's gateway, keeping the strongest implementation of each.

One caveat applies throughout: shell replies are demultiplexed per-request by `ConKernelClient`, but iopub is a single broadcast stream. `iopub_drain` filters it by parent msg_id and *discards* what it skips, so run one drain at a time. Concurrent executes are fine; concurrent *drains* are not.

## Imports

In [ ]:
#| export
from fastcore.utils import *
from conkernelclient.core import ConKernelClient, ConKernelManager
from contextlib import asynccontextmanager
from ast import literal_eval
from queue import Empty
import asyncio, time

In [ ]:
from fastcore.test import test_eq, test_fail
from jupyter_client.session import Session

In [ ]:
km = ConKernelManager(session=Session(key=b'x'))
await km.start_kernel()
kc = await km.client().start_channels()
await kc.is_alive()

True

## Message basics

In [ ]:
#| export
default_timeout = 10

def parent_id(msg):
    "The `msg_id` of the request this `msg` responds to, or None"
    return nested_idx(msg, "parent_header", "msg_id") or None

def iter_timeout(timeout=None, default=default_timeout):
    "Yield remaining seconds until `timeout` expires, using monotonic time"
    end = time.monotonic() + (timeout or default)
    while (rem := end - time.monotonic()) > 0: yield rem

Every reply carries its request's `msg_id` in the parent header; all routing below keys on it. `iter_timeout` drives every wait loop in this module: it yields the time still available, so a loop body can pass a shrinking timeout to each blocking call.

In [ ]:
r = await kc.execute('1+1', reply=True)
test_eq(parent_id(r), r['parent_header']['msg_id'])
assert parent_id({}) is None

## Iopub collection

In [ ]:
#| export
@patch
async def iopub_drain(self:ConKernelClient, msg_id, timeout=default_timeout):
    "Collect iopub messages parented to `msg_id` until its idle status arrives. Other requests' messages are discarded: one drain at a time"
    res = []
    for rem in iter_timeout(timeout):
        try: msg = await self.get_iopub_msg(timeout=rem)
        except Empty: continue
        if parent_id(msg) != msg_id: continue
        res.append(msg)
        if msg.get('msg_type') == 'status' and nested_idx(msg, 'content', 'execution_state') == 'idle': return res
    raise TimeoutError(f"iopub idle not seen for {msg_id}")

In [ ]:
#| export
@patch
async def iopub_flush(self:ConKernelClient, timeout=0.1):
    "Discard all pending iopub messages, e.g. leftovers from fire-and-forget executes"
    try:
        while True: await self.get_iopub_msg(timeout=timeout)
    except Empty: pass

`iopub_flush` is the counterpart for when you *don't* want collection: clear the backlog before starting a push-based consumer or a fresh interaction.

In [ ]:
kc.execute('print("junk")')
await kc.iopub_flush()
try:
    await kc.get_iopub_msg(timeout=0.1)
    assert False, 'iopub should be empty after flush'
except Empty: pass

Collecting a request's output means draining iopub until the kernel publishes the `idle` status for that request - that is the protocol's "this request is done publishing" signal, far more reliable than sweeping whatever is pending after a fixed delay.

In [ ]:
mid = kc.execute('print("hi"); 42')
msgs = await kc.iopub_drain(mid)
test_eq([m['msg_type'] for m in msgs if m['msg_type']!='status'], ['execute_input', 'stream', 'execute_result'])

## Outputs

In [ ]:
#| export
output_types = {'stream', 'execute_result', 'display_data', 'error'}

def nb_outputs(msgs):
    "Convert iopub `msgs` to nbformat-style output dicts, dropping non-output messages"
    res = []
    for m in msgs:
        if (mt := m['msg_type']) not in output_types: continue
        d = dict(output_type=mt, **m['content'])
        d.pop('transient', None)
        res.append(d)
    return res

In [ ]:
#| export
def iopub_msgs(msgs, msg_type=None):
    "Filter iopub `msgs` by `msg_type` - a single type, or a collection of types (all messages if None)"
    if msg_type is None: return msgs
    types = {msg_type} if isinstance(msg_type, str) else msg_type
    return [m for m in msgs if m['msg_type'] in types]

def iopub_streams(msgs, name=None):
    "The stream messages in `msgs`, optionally only stream `name` ('stdout'/'stderr')"
    streams = iopub_msgs(msgs, 'stream')
    return streams if name is None else [m for m in streams if m['content'].get('name') == name]

In [ ]:
test_eq([m['msg_type'] for m in iopub_msgs(msgs, 'execute_result')], ['execute_result'])
test_eq([m['msg_type'] for m in iopub_msgs(msgs, output_types)], ['stream', 'execute_result'])
test_eq(iopub_streams(msgs)[0]['content']['text'], 'hi\n')
test_eq(iopub_streams(msgs, 'stderr'), [])

Raw-message filters for when the protocol wrapping *is* wanted (parent ids, per-message metadata) - `nb_outputs` above is the lossy convenience, these are the lossless ones.

Iopub messages carry protocol wrapping (headers, status chatter, `execute_input` echoes) that consumers rarely want. `nb_outputs` reduces them to the same output dicts a notebook file stores, so downstream code (e.g. renderers expecting nbformat shapes) can consume them directly. This mirrors `nbformat.v4.output_from_msg` without the dependency, minus schema validation.

In [ ]:
outs = nb_outputs(msgs)
test_eq(outs[0], dict(output_type='stream', name='stdout', text='hi\n'))
test_eq(outs[1]['data']['text/plain'], '42')

## Execute composites

In [ ]:
#| export
@patch
async def exec_drain(self:ConKernelClient, code, timeout=default_timeout, **kw):
    "Execute `code`; return `(reply, outputs)` where outputs are the request's iopub messages"
    reply = await self.execute(code, reply=True, timeout=timeout, **kw)
    return reply, await self.iopub_drain(parent_id(reply), timeout=timeout)

@patch
async def exec_ok(self:ConKernelClient, code, timeout=default_timeout, **kw):
    "`exec_drain`, asserting the reply status is ok"
    reply, outputs = await self.exec_drain(code, timeout=timeout, **kw)
    assert reply['content']['status'] == 'ok', reply['content']
    return reply, outputs

@patch
async def exec_outs(self:ConKernelClient, code, timeout=default_timeout, **kw):
    "Execute `code` and return just its nbformat-style outputs"
    reply, outputs = await self.exec_drain(code, timeout=timeout, **kw)
    return nb_outputs(outputs)

The composites cover the three common shapes: full protocol detail (`exec_drain`), tests that just need success (`exec_ok`), and consumers that only want outputs (`exec_outs`).

In [ ]:
reply, outputs = await kc.exec_ok('x = 3; x')
test_eq(reply['content']['status'], 'ok')
test_eq((await kc.exec_outs('print(x*2)')), [dict(output_type='stream', name='stdout', text='6\n')])
reply, _ = await kc.exec_drain('1/0')
test_eq(reply['content']['ename'], 'ZeroDivisionError')

## Expression values

In [ ]:
#| export
class EvalError(Exception):
    "An `eval_expr` expression raised in the kernel"

def parse_expr(s):
    "The `literal_eval` of `s` when its form allows, else `s` unchanged"
    try: return literal_eval(s)
    except Exception: return s

@patch
async def user_exprs(self:ConKernelClient, exprs:dict, code:str='', timeout=default_timeout, **kw):
    "Run `code` and evaluate each of the `exprs` expressions in the same round trip; returns the reply content (statuses and mimebundles unparsed)"
    return (await self.execute(code, reply=True, timeout=timeout, user_expressions=exprs, store_history=False, **kw))['content']

@patch
async def eval_expr(self:ConKernelClient, expr:str, code:str='', timeout=default_timeout, **kw):
    "Evaluate `expr` in the kernel (optionally after running `code`) and return its value: parsed via `literal_eval` when its repr allows, else the repr string. Raises `EvalError` if the kernel raises"
    cts = await self.user_exprs({'_': expr}, code, timeout=timeout, **kw)
    if cts['status'] != 'ok': raise EvalError(f"{cts.get('ename')}: {cts.get('evalue')}")
    r = cts['user_expressions']['_']
    if r.get('status') != 'ok': raise EvalError(f"{r.get('ename')}: {r.get('evalue')}")
    return parse_expr(r['data']['text/plain'])

The `user_expressions` round trip: one execute carries the expression, the kernel evaluates it *after* the (empty) cell and returns its repr inside the execute_reply - no iopub involved, so it works cleanly alongside streaming output. Values whose repr is not `literal_eval`-able come back as that repr string. This is the general core of solveit's richer `eval` RPC.

In [ ]:
await kc.exec_ok("v = [1, 'a', {'b': 2}]")
test_eq(await kc.eval_expr('v'), [1, 'a', {'b': 2}])
test_eq(await kc.eval_expr('w * 2', code='w = 21'), 42)
cts = await kc.user_exprs({'a': 'v[0]', 'b': 'len(v)'})
test_eq(cts['status'], 'ok')
test_eq(parse_expr(cts['user_expressions']['a']['data']['text/plain']), 1)
test_eq(parse_expr(cts['user_expressions']['b']['data']['text/plain']), 3)
test_eq(await kc.eval_expr('sum(v[2].values())'), 2)
r = await kc.eval_expr('print')
assert isinstance(r, str) and 'print' in r, r
try:
    await kc.eval_expr('nope_undefined')
    assert False, 'expected EvalError'
except EvalError as e: assert 'NameError' in str(e)

## Generic requests

In [ ]:
#| export
@patch
def shell_request(
    self:ConKernelClient, msg_type, timeout=default_timeout, reply=True, buffers=None,
    msg_id=None, # Override the auto-generated message id
    subshell_id=None, # Route via this subshell (header field)
    metadata=None, # Message metadata (e.g. ipywidgets control-comm version check)
    **content
):
    "Send an arbitrary shell request, routed through the reply reader like `execute`. Returns a coroutine for the reply, or just the msg_id when `reply=False` (for fire-and-forget types like `comm_open` that never get replies)"
    if not self._check_alive(): return asyncio.sleep(0) if reply else None
    msg = self.session.msg(msg_type, content, metadata=metadata)
    if msg_id is not None: msg['header']['msg_id'] = msg_id
    if subshell_id is not None: msg['header']['subshell_id'] = subshell_id
    mid = msg['header']['msg_id']
    if reply: self._pending[mid] = (asyncio.Queue(maxsize=1), False)
    if buffers: self.session.send(self.shell_channel.socket, msg, buffers=buffers)
    else: self.shell_channel.send(msg)
    return self._async_recv_reply(mid, timeout=timeout) if reply else mid

@patch
async def control_request(self:ConKernelClient, msg_type, timeout=default_timeout, **content):
    "Send a request on the control channel and await its matching reply"
    msg = self.session.msg(msg_type, content)
    self.control_channel.send(msg)
    for rem in iter_timeout(timeout):
        try: reply = await self.get_control_msg(timeout=rem)
        except Empty: continue
        if parent_id(reply) == msg['header']['msg_id']: return reply
    raise TimeoutError(f"no reply to {msg_type}")

`ConKernelClient`'s background reader consumes *every* shell message, so a bare `get_shell_msg` would race it and lose. Any non-execute shell request (`kernel_info_request`, `complete_request`, ...) must therefore register with the same routing table `execute` uses - that is what `shell_request` does. Pass `reply=False` for fire-and-forget message types the kernel never answers (`comm_open`, `comm_msg`, `comm_close`), and `buffers` for binary payloads. Control-channel replies have no reader competing for them, so `control_request` matches by parent id directly. Note control requests should not be issued concurrently with each other.

In [ ]:
r = await kc.shell_request('kernel_info_request')
test_eq(r['header']['msg_type'], 'kernel_info_reply')
r = await kc.control_request('kernel_info_request')
test_eq(r['header']['msg_type'], 'kernel_info_reply')

Read the `input_request` from the stdin channel with `get_stdin_msg`, then answer it; the blocked execute's reply follows.

In [ ]:
#| export
@patch
async def input_reply(self:ConKernelClient, value:str):
    "Answer the kernel's pending input_request (from `input()` in an execute sent with `allow_stdin=True`)"
    await self.stdin_send(self.session.msg('input_reply', dict(value=value)))

In [ ]:
c = kc.execute("x = input('name? ')", reply=True, timeout=10, allow_stdin=True)
msg = await kc.get_stdin_msg(timeout=10)
test_eq(msg['header']['msg_type'], 'input_request')
await kc.input_reply('Ada')
test_eq((await c)['content']['status'], 'ok')
test_eq((await kc.exec_outs('x'))[0]['data']['text/plain'], "'Ada'")

In [ ]:
code = """def _t(comm, msg):
    global nbuf; nbuf = len(msg.get('buffers', []))
get_ipython().kernel.comm_manager.register_target('t', _t)
"""
await kc.exec_ok(code)
mid = kc.shell_request('comm_open', reply=False, comm_id='c1', target_name='t', data={}, buffers=[b'12345'])
assert isinstance(mid, str)
await kc.exec_ok('import time; time.sleep(0.2)')
test_eq((await kc.exec_outs('nbuf'))[0]['data']['text/plain'], '1')


In [ ]:
code = """def _tm(comm, msg):
    global nmd; nmd = msg.get('metadata')
get_ipython().kernel.comm_manager.register_target('tm', _tm)
"""
await kc.exec_ok(code)
kc.shell_request('comm_open', reply=False, comm_id='cm1', target_name='tm', data={}, metadata={'version':'2.1.0'})
await kc.exec_ok('import time; time.sleep(0.2)')
test_eq((await kc.exec_outs('nmd'))[0]['data']['text/plain'], "{'version': '2.1.0'}")

## Proxies

In [ ]:
#| export
class _ReqProxy:
    "Attribute access to protocol requests: `kc.cmd.complete(code=...)` sends `complete_request` on shell"
    def __init__(self, kc, send, suffix='_request'): self.kc,self.send,self.suffix = kc,send,suffix
    def __getattr__(self, name):
        if name.startswith('_'): raise AttributeError(name)
        def _call(*, timeout=default_timeout, **content): return self.send(f"{name}{self.suffix}", timeout=timeout, **content)
        return _call

@patch(as_prop=True)
def cmd(self:ConKernelClient):
    "Shell requests as methods: `await kc.cmd.history(...)` sends `history_request`"
    return _ReqProxy(self, self.shell_request)

@patch(as_prop=True)
def ctl(self:ConKernelClient):
    "Control requests as methods: `await kc.ctl.shutdown(restart=True)` sends `shutdown_request`"
    return _ReqProxy(self, self.control_request)

In [ ]:
#| export
class _DapProxy:
    "DAP requests as methods: `kc.dap.stackTrace(threadId=1)` sends a `debug_request` on control"
    def __init__(self, kc): self.kc,self.seq = kc,0
    def __getattr__(self, command):
        if command.startswith('_'): raise AttributeError(command)
        if command.endswith('_') and not command.endswith('__'): command = command[:-1]
        async def _call(*, timeout=default_timeout, full=False, **arguments):
            self.seq += 1
            r = await self.kc.control_request('debug_request', timeout=timeout,
                type='request', seq=self.seq, command=command, arguments=arguments)
            return r if full else r['content']
        return _call

@patch(as_prop=True)
def dap(self:ConKernelClient):
    "DAP debug requests as methods. Stateful (`seq` counter), so cached per client, unlike `cmd`/`ctl`"
    if getattr(self, '_dap', None) is None: self._dap = _DapProxy(self)
    return self._dap

`dap` speaks the Debug Adapter Protocol over `debug_request` control messages. A trailing underscore escapes Python keywords (`kc.dap.continue_(...)` sends the DAP `continue` command); `full=True` returns the whole reply message instead of just its content.

The proxies make one-off protocol requests read like methods without hand-assembling messages. They are stateless, so no caching is needed.

In [ ]:
r = await kc.cmd.complete(code='pri', cursor_pos=3)
assert 'print' in r['content']['matches']
r = await kc.cmd.is_complete(code='def f():')
test_eq(r['content']['status'], 'incomplete')

In [ ]:
r = await kc.shell_request('kernel_info_request', msg_id='fixed-id-1')
test_eq(parent_id(r), 'fixed-id-1')
r = await kc.dap.debugInfo()
test_eq(r['command'], 'debugInfo')
assert r['success']
r2 = await kc.dap.debugInfo(full=True)
test_eq(parent_id(r2), parent_id(r2))  # full message shape
assert r2['header']['msg_type'] == 'debug_reply' 

## Interrupt

In [ ]:
#| export
@patch
async def interrupt(self:ConKernelClient, timeout=5):
    "Interrupt the running request via `interrupt_request` on control; returns the reply"
    return await self.control_request('interrupt_request', timeout=timeout)

The interrupted `execute` returns an error reply with `ename` `KeyboardInterrupt`, and the kernel stays usable. (Solveit's gateway historically followed the interrupt with an empty `execute` to wake a parked recv; the `Session.send` patch in `core` addresses that wake-up at the transport level, so it is not repeated here.)

In [ ]:
task = asyncio.ensure_future(kc.execute('import time; time.sleep(30)', reply=True, timeout=15))
await asyncio.sleep(0.5)
r = await kc.interrupt()
test_eq(r['header']['msg_type'], 'interrupt_reply')
reply = await task
test_eq(reply['content']['ename'], 'KeyboardInterrupt')
test_eq((await kc.exec_outs('40+2'))[0]['data']['text/plain'], '42')

## Lifecycle

In [ ]:
#| export
@asynccontextmanager
async def run_kernel(
    kernel_name='python3', # Kernelspec name to launch
    manager_cls=ConKernelManager, # Manager class, e.g. a subclass customizing launch
    **kwargs # Passed to `start_kernel` (e.g. `env`, `cwd`)
):
    "Start a kernel, yield `(km, kc)` with channels running, and always shut both down"
    km = manager_cls(kernel_name=kernel_name)
    await km.start_kernel(**kwargs)
    kc = await km.client().start_channels()
    try: yield km, kc
    finally:
        kc.stop_channels()
        await km.shutdown_kernel(now=True)

In [ ]:
#| export
async def reconnect(km, kc=None):
    "Restart `km`'s kernel (fresh process, state discarded) and return a newly connected client, stopping `kc` if given"
    if kc is not None: kc.stop_channels()
    await km.restart_kernel(now=True)
    return await km.client().start_channels()

In [ ]:
async with run_kernel() as (km3, kc3):
    await kc3.exec_ok('x = 42')
    pid1 = km3.provisioner.pid
    kc3 = await reconnect(km3, kc3)
    assert km3.provisioner.pid != pid1
    reply, _ = await kc3.exec_drain('x')
    test_eq(reply['content']['ename'], 'NameError')
    test_eq((await kc3.exec_outs('1+1'))[0]['data']['text/plain'], '2')

After a restart the old client's channels and routed-reply reader are stale, so `reconnect` hands back a fresh client bound to the restarted process. State is gone: redo any imports/setup.

The context manager guarantees teardown even when a test body raises, which is what keeps a big protocol test suite from leaking kernel processes.

In [ ]:
async with run_kernel() as (km2, kc2):
    test_eq((await kc2.exec_outs('1+1'))[0]['data']['text/plain'], '2')
    pid = km2.provisioner.pid
assert not await km2.is_alive()

## Cleanup

In [ ]:
if await km.is_alive():
    kc.stop_channels()
    await km.shutdown_kernel()

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()